# Experiment 10: Text-to-Video Generation
**Aim:** Generate a short video from a natural-language prompt using a pretrained model. No dataset collection or model training is required.

**Compute note:** This model is memory-intensive. GPU is strongly recommended; availability and compatibility vary.

In [ ]:
%pip install -q diffusers transformers accelerate imageio imageio-ffmpeg

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 1. Load the pretrained text-to-video pipeline
The first load downloads model weights and may take several minutes.

In [ ]:
from diffusers import DiffusionPipeline
import torch, imageio
from pathlib import Path
from IPython.display import Video, display
model_id = 'damo-vilab/text-to-video-ms-1.7b'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype)
if device == 'cuda':
    try:
        pipe.enable_model_cpu_offload()
    except Exception:
        pipe = pipe.to(device)
else:
    pipe = pipe.to(device)
print('Pipeline loaded on', device)

## 2. Enter a prompt and generate frames
Keep the prompt concise. The generation can take time and may run out of memory on limited GPUs.

In [ ]:
prompt = 'A small boat sailing across a calm lake at sunrise, gentle water movement, cinematic wide shot'
with torch.inference_mode():
    result = pipe(prompt, num_frames=16, num_inference_steps=25)
frames = result.frames[0]
print('Generated frames:', len(frames))

## 3. Save video as MP4

In [ ]:
out_dir = Path('outputs'); out_dir.mkdir(exist_ok=True)
video_path = out_dir / 'generated_video.mp4'
imageio.mimsave(video_path, frames, fps=8, macro_block_size=1)
print('Saved:', video_path)

## 4. Preview and download

In [ ]:
display(Video(str(video_path), embed=True))
from google.colab import files as colab_files
colab_files.download(str(video_path))

## 5. Experiment with another prompt (optional)
Change the prompt to a different scene or visual style and rerun the generation and save cells.

In [ ]:
prompt2 = 'A red sports car driving along a coastal road at golden hour, smooth tracking shot, cinematic style'
with torch.inference_mode():
    result2 = pipe(prompt2, num_frames=16, num_inference_steps=25)
frames2 = result2.frames[0]
video_path2 = out_dir / 'generated_video_2.mp4'
imageio.mimsave(video_path2, frames2, fps=8, macro_block_size=1)
display(Video(str(video_path2), embed=True))
print('Saved:', video_path2)